# 01 · Reddit data audit

**Goal:** understand the raw Reddit pull before trusting anything downstream — coverage, volume by subreddit, time span, and whether we are looking at real or synthetic (mock) data. Bad inputs silently poison every feature and backtest, so we audit first.

Run `make fetch-reddit` first (mock works with no keys).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd, numpy as np
from reddit_hype.config import load_settings, credential_report
from reddit_hype.utils import read_parquet_or_empty
s = load_settings()
print(credential_report())

In [ ]:
items = read_parquet_or_empty(s.path('reddit_items'))
print('rows:', len(items), '| synthetic:', bool(items.get('synthetic', pd.Series([False])).max()))
items['created_dt'] = pd.to_datetime(items['created_dt'], utc=True)
print('span:', items['created_dt'].min(), '->', items['created_dt'].max())
items.head(3)

### Volume by subreddit and post/comment split

In [ ]:
display(items.groupby(['subreddit','kind']).size().unstack(fill_value=0))
items['kind'].value_counts()

### Daily activity over time

In [ ]:
daily = items.assign(d=items['created_dt'].dt.date).groupby('d').size()
daily.plot(title='Reddit items per day', figsize=(10,3)); None

### Data-quality flags
If `synthetic` is True, these are random development data — do **not** read any signal into them. Provide `REDDIT_*` credentials and re-fetch for real analysis.